# Experimentation with GEC into the full pipeline


In [1]:
import json
import logging
import os
import sys

# NOTE: import from root of the project
sys.path.append(os.path.abspath("../../../../"))

from src.services.gec.modules.dictionary.engine import DictionaryEngine
from src.services.gec.schemas import DictionaryCandidateEdit, GECInput, ModuleStatus
from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("src.services.gec.modules.dictionary.spell_checker").setLevel(
    logging.DEBUG
)

In [2]:
dictionary_engine = DictionaryEngine()

2026-06-19 18:47:51.713 | INFO     | src.services.gec.modules.dictionary.arramooz_client:__init__:36 - ArramoozClient initialized | dict_db=/Users/incorta/College/GP/baligh/src/services/gec/data/dictionary/arabicdictionary.sqlite
2026-06-19 18:47:51.761 | INFO     | src.services.gec.modules.dictionary.arramooz_client:get_all_normalized_words:123 - Fetched 31190 normalized words from dictionary
2026-06-19 18:47:51.768 | INFO     | src.services.gec.modules.dictionary.spell_checker:__init__:28 - SpellChecker initialized | vocabulary_size=31190
2026-06-19 18:47:51.769 | INFO     | src.services.gec.modules.dictionary.engine:__init__:35 - DictionaryEngine initialized successfully


In [3]:
def test_dictionary(
    text: str, sum_output: bool = True, show_preprocessing: bool = False
):
    """Tests dictionary-based GEC with preprocessing."""
    pre_input = PreprocessingInput(text=text)
    pre_output = preprocess(pre_input)

    gec_input = GECInput(
        text=pre_output.text,
        tokens=pre_output.tokens,
        morph_features=pre_output.morph_features,
        errors_span=[],
    )
    gec_output = dictionary_engine.process(gec_input)

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    if sum_output:
        if gec_output.status == ModuleStatus.CORRECT:
            print("No errors found")
        else:
            print(f"Found {len(gec_output.candidate_edits)} candidate edits:")
            for i in range(len(gec_output.candidate_edits)):
                edit: DictionaryCandidateEdit = gec_output.candidate_edits[i]
                print("*" * 20 + f" Edit {i} " + "*" * 20)
                print(f"  Error in: {pre_output.tokens[edit.token_refs[0]]}")
                print(f"  Corrected text: {edit.correction}")
                print(f"  Alternative texts: {sorted(edit.alternatives, reverse=True)}")
    else:
        print(json.dumps(gec_output.model_dump(), ensure_ascii=False, indent=2))

---
### Open testing

In [4]:
text = "ذهب محمد إلى المدلرسة. القتاب علي الطاوولة. تعبت حتي تعلمت."
test_dictionary(text, sum_output=True)

[2026-06-19 18:47:52,147 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
2026-06-19 18:48:02.036 | INFO     | src.services.gec.modules.dictionary.engine:process:56 - DictionaryEngine.process | tokens=13 errors_span=0
2026-06-19 18:48:02.037 | DEBUG    | src.services.gec.modules.dictionary.engine:process:74 - GED-flagged orthography tokens: 0
2026-06-19 18:48:02.528 | DEBUG    | src.services.gec.modules.dictionary.spell_checker:generate_candidates:120 - Total candidates for 'المدلرسة': 18
2026-06-19 18:48:02.529 | DEBUG    | src.services.gec.modules.dictionary.alternative_ranker:rank_alternatives:60 - Ranking 18 candidates for 'المدلرسة'
2026-06-19 18:48:02.529 | DEBUG    | src.services.gec.modules.dictionary.arramooz_client:_ensure_freq_db:133 - Lazily opening frequency DB | path=/Users/incorta/College/GP/baligh/src/services/gec/data/dictionary/wordfreq.sqlite
2026-06-19 18:48:02.538 | DE

Found 4 candidate edits:
******************** Edit 0 ********************
  Error in: index=3 form='المدلرسة' span=(13, 21) norm_span=(13, 21) affix_structure='DET+STEM' is_oov=True
  Corrected text: المدرسة
  Alternative texts: ['الممارسة', 'المرسة', 'المديرية', 'المدلس', 'المدرعة', 'المدرسي', 'المدرسة', 'المدرس', 'المدارسة', 'المداراة']
******************** Edit 1 ********************
  Error in: index=5 form='القتاب' span=(23, 29) norm_span=(23, 29) affix_structure='DET+STEM' is_oov=True
  Corrected text: الكتاب
  Alternative texts: ['الكتاب', 'الكتاب', 'الكتاب', 'القطاع', 'القرار', 'القرار', 'القتال', 'القتال', 'القال', 'القال']
******************** Edit 2 ********************
  Error in: index=7 form='الطاوولة' span=(34, 42) norm_span=(34, 42) affix_structure='DET+STEM' is_oov=True
  Corrected text: الطاولة
  Alternative texts: ['الطفولة', 'الطاووس', 'الطاولة', 'الطاول', 'الحامولة']
******************** Edit 3 ********************
  Error in: index=10 form='حتي' span=(49, 52) norm